In [3]:
# -*- coding: utf-8 -*-
"""Apexify_Technologys_FAQ_Chatbot_Preparation.ipynb

# Apexify Technologys - FAQ Data Preparation
Run this in Google Colab to generate the FAQ data for the Apexify Technologys website chatbot.
"""

import pandas as pd
import numpy as np
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
import warnings
warnings.filterwarnings('ignore')

# Download necessary NLTK data
print("Downloading NLTK data...")
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print("NLTK data downloaded.")

# --- UPDATED & EXPANDED FAQ DATA ---
# Based on the website: https://professionaltechcompany-website-noura.netlify.app/#home

# First, create separate lists to ensure we can count them easily
questions = [
    # Basic & Company Info (NEW)
    "Hello", "Hi", "Hey", "What's up",                     # Greetings
    "Who are you?", "Tell me about yourself",              # Company Identity
    "What is Apexify Technologys?",
    "What is your vision?",                                 # From website
    "What is your mission?",                                # From website
    "What makes you different?",
    "What do you do?",
    "How can I contact you?", "Contact information",       # Contact

    # Internships & Programs (EXPANDED based on website)
    "Tell me about your internship program",                # Core offering
    "Do you offer internships?",
    "What internships are available?",
    "Frontend Development internship",                      # Specific domains
    "Backend Development internship",
    "App Development internship",
    "Artificial Intelligence internship",
    "Data Science internship",
    "Digital Marketing internship",
    "Graphic Designing internship",
    "How do I apply for an internship?",                    # Application process
    "What is the application process?",
    "Where do I submit my resume?",
    "I want to apply, what should I do?",
    "Are the internships paid or unpaid?",
    "Who can apply for an internship?",                     # Eligibility
    "Is the internship for students only?",
    "What skills do I need for an internship?",

    # Services (ALIGNED with website)
    "What services do you offer?",
    "Do you provide IT services?",                           # IT Services & Consulting
    "What is your skills development program?",              # Skills Development
    "How can I improve my tech skills?",
    "Do you offer consulting?",

    # General AI & Tech Information (NEW - Educational)
    "Tell me about Artificial Intelligence", "What is AI?", # General AI info
    "What is machine learning?",
    "What is data science?",                                 # Related to internship
    "What do frontend developers do?",
    "What is digital marketing?",
    "What is graphic design?",

    # Process & Logistics
    "How long is the internship?",
    "Is the internship remote or in-office?",
    "Do I get a certificate after the internship?",
    "Will I get a job after the internship?",
    "Do you provide mentorship?",                            # From mission statement
    "What projects will I work on?",
    "What technologies will I learn?",

    # Misc
    "Do you have any success stories?",                      # Case studies idea
    "What are your business hours?"
]

answers = [
    # Basic & Company Info (NEW)
    "Hello! Welcome to Apexify Technologys. How can I help you with your career journey today?",  # Hello
    "Hi there! I'm the Apexify assistant. I'm here to answer your questions about our internships and programs.", # Hi
    "Hey! Great to hear from you. What would you like to know about Apexify Technologys?", # Hey
    "Not much, just here to help you explore opportunities at Apexify! How can I assist?", # What's up
    "I'm the virtual assistant for Apexify Technologys, an information services company dedicated to helping people polish their skills and gain experience in tech fields.", # Who are you
    "Apexify Technologys is an information services company that empowers aspiring professionals by bridging the gap between academic knowledge and industry requirements through practical internships and skill development.", # Tell me about yourself
    "Apexify Technologys is a platform dedicated to helping individuals polish their skills and gain practical experience in various tech fields like software development, data science, and marketing to launch their careers.", # What is Apexify
    "Our vision is to become a leading provider of internships and job opportunities in the tech industry, creating a community of talented and motivated individuals passionate about innovation and excellence.", # Vision
    "Our mission is to provide high-quality internships and job opportunities. We are committed to offering guidance and mentorship, fostering innovation and teamwork, building strong industry relationships, and rewarding outstanding performance.", # Mission
    "Apexify stands out by focusing on practical, hands-on experience. We connect your academic knowledge with real-world industry requirements through dedicated mentorship and projects, ensuring you're job-ready.", # What makes you different
    "We help people polish their skills and get experience in their chosen fields. We offer a comprehensive internship program across multiple domains like AI, Data Science, and Development, along with IT services and skills development.", # What do you do
    "You can reach us through the Contact form on our website. Just submit your inquiry or resume there, and our team will get back to you promptly. For specific questions, you can also email us.", # Contact
    "You can reach us through the Contact form on our website. Just submit your inquiry or resume there, and our team will get back to you promptly. For specific questions, you can also email us.", # Contact information (duplicate answer for "Contact information")

    # Internships & Programs (EXPANDED)
    "Our internship program is designed to give you hands-on experience in areas like software development, data science, and marketing. You'll work on real-world projects, collaborate with experienced professionals, and develop valuable skills highly sought after by employers.", # Internship program
    "Yes, absolutely! We offer a comprehensive internship program for students and recent graduates looking to kickstart their careers in tech. You can apply via our website's Contact form.", # Do you offer
    "We offer internships in several exciting fields: Frontend Development, Backend Development, App Development, Artificial Intelligence, Data Science, Digital Marketing, and Graphic Designing.", # What internships
    "The Frontend Development internship focuses on building responsive and interactive user interfaces using modern web technologies like HTML, CSS, and JavaScript frameworks. You'll learn to create engaging user experiences.", # Frontend
    "In the Backend Development internship, you'll learn to build robust server-side applications, APIs, and databases to power web and mobile applications. You'll work with languages like Python, Node.js, and database systems.", # Backend
    "The App Development internship lets you dive into mobile app development, primarily for Android. You'll learn the fundamentals of mobile UI/UX, working with native features to build functional and appealing apps.", # App Dev
    "The Artificial Intelligence internship lets you explore the world of AI and machine learning. You'll work on projects involving data analysis, model building, and understanding algorithms to solve real-world problems.", # AI internship
    "As a Data Science intern, you'll develop skills in data collection, cleaning, analysis, and visualization. The goal is to uncover insights from data and help make data-driven decisions, using tools like Python and its data libraries.", # Data Science
    "In the Digital Marketing internship, you'll learn strategies for online marketing, including Search Engine Optimization (SEO), content creation, social media campaigns, and analytics to understand digital audience behavior.", # Digital Marketing
    "The Graphic Designing internship will help you master design principles and software (like Adobe Suite or Figma) to create visually appealing graphics for web and print materials, building a strong portfolio.", # Graphic Designing
    "To apply, simply submit your resume through our Contact form on the website. Navigate to the #contact section and fill out the form with your details and attach your resume. We look forward to hearing from you!", # How to apply
    "The application process is simple: 1. Prepare your updated resume. 2. Go to our website's Contact form. 3. Fill in your details and attach your resume. 4. Submit the form. Our team will review it and contact you for the next steps.", # Application process
    "Please submit your resume via our Contact form on the website. That's the primary and best way for us to receive your application and information.", # Submit resume
    "Great! First, make sure your resume is up-to-date. Then, head over to our website's Contact form, fill in all the required information, attach your resume, and send it. Our recruitment team will be in touch!", # What should I do
    "Some of our internship positions may offer stipends or performance-based pay, but it's best to check the specific details for each internship posting or ask during the interview process. Many are focused on providing high-quality learning experience.", # Paid/Unpaid
    "Our internships are primarily designed for students and recent graduates who are looking to kickstart their career in tech. However, we encourage anyone passionate about learning and gaining practical experience to apply and showcase their motivation.", # Who can apply
    "While our program is ideal for students and recent graduates, we consider all applicants who demonstrate a strong passion for technology and a willingness to learn. Don't hesitate to apply if you're motivated!", # Students only
    "The required skills vary by domain. Generally, a foundational understanding of the field (e.g., basic programming for dev roles, familiarity with marketing concepts for Digital Marketing) and a strong eagerness to learn are key. Check the specific internship description for details.", # Skills needed

    # Services (ALIGNED)
    "We offer three main services: 1. A hands-on **Internship Program** in various tech fields. 2. **IT Services & Consulting** to help businesses achieve their technology goals. 3. **Skills Development** to help individuals polish their skills in areas like programming and lead generation.", # Services
    "Yes, we provide expert IT services and consulting to help businesses achieve their technology goals. This is separate from our internship programs and focuses on solving business challenges with technology.", # IT Services
    "Our Skills Development program is designed to help individuals polish their skills and gain real-world experience in specific tech fields. This might involve focused workshops, projects, or training in programming languages, lead generation, and other practical areas.", # Skills Development
    "You can improve your tech skills by joining our internship program or Skills Development initiatives. These programs offer hands-on projects and mentorship to help you build practical, job-ready skills in your chosen area.", # Improve skills
    "Yes, we offer IT Services & Consulting. Our team can provide expert advice and solutions to help businesses leverage technology effectively, whether it's for system architecture, process automation, or digital strategy.", # Consulting

    # General AI & Tech Information (NEW)
    "Artificial Intelligence (AI) is a broad field of computer science focused on creating systems capable of performing tasks that typically require human intelligence. These include learning, reasoning, problem-solving, perception, and language understanding. Our AI internship lets you work on real projects in this exciting field.", # What is AI (for "Tell me about Artificial Intelligence")
    "Artificial Intelligence (AI) is a broad field of computer science focused on creating systems capable of performing tasks that typically require human intelligence. These include learning, reasoning, problem-solving, perception, and language understanding. Our AI internship lets you work on real projects in this exciting field.", # What is AI (for "What is AI?")
    "Machine Learning (ML) is a subset of AI where computers learn from data without being explicitly programmed. Instead of following strict rules, ML algorithms identify patterns in data to make predictions or decisions. It's a key skill in our Data Science and AI internships.", # What is ML
    "Data Science is an interdisciplinary field that uses scientific methods, algorithms, and systems to extract knowledge and insights from structured and unstructured data. It combines statistics, data analysis, and machine learning to understand and analyze actual phenomena with data.", # What is Data Science
    "Frontend developers build the parts of a website or app that users see and interact with directly (the user interface). They use languages like HTML, CSS, and JavaScript to create responsive, interactive, and visually appealing experiences.", # What do frontend devs do
    "Digital Marketing encompasses all marketing efforts that use an electronic device or the internet. It includes channels like search engines (SEO/SEM), social media, email, and content marketing to connect with current and prospective customers.", # What is digital marketing
    "Graphic design is a craft where professionals create visual content to communicate messages. By applying visual hierarchy and page layout techniques, designers use typography and pictures to meet users' specific needs and focus on the logic of displaying elements in interactive designs.", # What is graphic design

    # Process & Logistics
    "The duration can vary depending on the specific internship and your availability. Typically, our internships are structured to last between 3 to 6 months, allowing enough time to work on meaningful projects and develop key skills.", # How long
    "This can depend on the specific role and current company policy. We offer both remote and in-office possibilities. This will be discussed and clarified during the interview process for the specific internship you are applying for.", # Remote/office
    "Yes, upon successful completion of the internship program, you will receive a certificate from Apexify Technologys. This certificate acknowledges your participation and the practical skills you've gained during your time with us.", # Certificate
    "While we cannot guarantee a job, our program is designed to make you highly employable. We focus on practical skills, real-world projects, and mentorship. High-performing interns may also get opportunities for freelance projects, recommendations, or consideration for future full-time roles within our partner network.", # Job after
    "Absolutely! As stated in our mission, we are committed to offering guidance and mentorship. You will have the opportunity to collaborate with experienced professionals who will help you achieve your career goals and navigate your projects.", # Mentorship
    "You'll work on real-world projects that simulate actual industry challenges. For example, a development intern might build a feature for a website, while a data science intern might analyze a dataset to find trends. The goal is to give you portfolio-ready experience.", # What projects
    "You'll learn technologies relevant to your chosen field. For instance, web development interns might learn HTML, CSS, JavaScript, React, Node.js, or Python. AI/Data Science interns might use Python, TensorFlow, or Pandas. The focus is on modern, in-demand tools.", # What tech

    # Misc
    "Yes, we have numerous success stories of past interns who have gone on to secure great jobs in the industry. We are building a collection of case studies and testimonials to showcase these journeys. Contact us if you'd like to speak with a former intern!", # Success stories
    "Our general business hours are Monday-Friday, 9 AM to 6 PM (your local time, depending on the office). However, our online application form is always open, and we strive to respond to inquiries as quickly as possible during the work week." # Business hours
]

# Verify lengths before creating the dictionary
print(f"Questions count: {len(questions)}")
print(f"Answers count: {len(answers)}")

# Check that questions and answers lists have the same length
assert len(questions) == len(answers), f"Questions and answers lists must have the same length! Questions: {len(questions)}, Answers: {len(answers)}"

# Create the FAQ data dictionary
faq_data = {
    'questions': questions,
    'answers': answers
}

print(f"✅ Created {len(faq_data['questions'])} FAQ pairs.")

# Create DataFrame
df = pd.DataFrame(faq_data)

# Save to CSV
df.to_csv('apexify_faqs.csv', index=False)
print("✅ FAQ data saved to 'apexify_faqs.csv'.")

# --- Text Preprocessing ---
def preprocess_text(text):
    """Clean and preprocess text for better matching"""
    if not isinstance(text, str):
        text = str(text)
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and digits (keep letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

# Apply preprocessing to questions
df['processed_questions'] = df['questions'].apply(preprocess_text)
print("✅ Text preprocessing complete.")

# --- Create TF-IDF Vectors for Similarity Search ---
# Create TF-IDF vectorizer with improved parameters
vectorizer = TfidfVectorizer(
    max_features=2000,  # Increased for better coverage
    stop_words='english',
    ngram_range=(1, 3),  # Use unigrams, bigrams, and trigrams for better matching
    min_df=1,  # Minimum document frequency
    max_df=0.9  # Ignore terms that appear in too many documents
)

# Fit and transform the questions
tfidf_matrix = vectorizer.fit_transform(df['processed_questions'])

print(f"✅ TF-IDF matrix created with shape: {tfidf_matrix.shape}")
print(f"✅ Vocabulary size: {len(vectorizer.vocabulary_)}")

# --- Save the Model and Data ---
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

df.to_pickle('faq_data.pkl')

print("\n✅ All models and data saved successfully:")
print("   - vectorizer.pkl")
print("   - tfidf_matrix.pkl")
print("   - faq_data.pkl")
print("   - apexify_faqs.csv")

# --- Test the Chatbot Function ---
def get_best_answer(user_question, vectorizer, tfidf_matrix, df, threshold=0.15):
    """
    Find the most similar FAQ question and return its answer

    Parameters:
    - user_question: str, the question from the user
    - vectorizer: trained TF-IDF vectorizer
    - tfidf_matrix: TF-IDF matrix of FAQ questions
    - df: DataFrame with questions and answers
    - threshold: float, minimum similarity score to return a match

    Returns:
    - answer: str, the best matching answer or fallback message
    - best_score: float, the similarity score
    - matched_question: str, the question that matched (for debugging)
    """
    # Preprocess user question
    processed_question = preprocess_text(user_question)

    # If question is empty after preprocessing
    if not processed_question:
        return "I didn't understand that. Could you please rephrase your question?", 0.0, ""

    # Vectorize user question
    user_vector = vectorizer.transform([processed_question])

    # Calculate cosine similarity
    similarities = cosine_similarity(user_vector, tfidf_matrix).flatten()

    # Get best match
    best_idx = similarities.argmax()
    best_score = similarities[best_idx]

    if best_score >= threshold:
        return df.iloc[best_idx]['answers'], best_score, df.iloc[best_idx]['questions']
    else:
        fallback_message = ("I'm not sure about that specific detail. For the most accurate information, "
                           "please contact us directly through the website's contact form or check the "
                           "internships section. You can also try rephrasing your question.")
        return fallback_message, best_score, ""

# --- Run a Comprehensive Test ---
print("\n" + "="*70)
print("TESTING THE CHATBOT WITH SAMPLE QUESTIONS")
print("="*70)

test_questions = [
    "Hello there",                           # Basic greeting
    "What internships do you have?",          # Core offering
    "Tell me about the AI internship",        # Specific domain
    "How do I apply?",                        # Process
    "What is your mission?",                   # Company info
    "What is machine learning?",               # General info
    "Do you pay interns?",                     # Variation of paid/unpaid
    "Can I work from home?",                   # Remote work
    "This is a totally random question"       # Low-score test
]

print("\n{:<40} {:<10} {:<30}".format("QUESTION", "SCORE", "MATCHED QUESTION"))
print("-"*80)

for q in test_questions:
    answer, score, matched_q = get_best_answer(q, vectorizer, tfidf_matrix, df)

    # Truncate for display
    q_display = q[:38] + "..." if len(q) > 38 else q
    matched_display = matched_q[:28] + "..." if len(matched_q) > 28 else matched_q

    print(f"{q_display:<40} {score:.3f}     {matched_display:<30}")

    # Print full answer for first few questions
    if test_questions.index(q) < 3:
        print(f"  Answer: {answer[:150]}...")
        print()

print("\n" + "="*70)

# --- Analyze FAQ Coverage ---
print("\n📊 FAQ COVERAGE ANALYSIS")
print("-"*40)
print(f"Total FAQ pairs: {len(df)}")
print(f"Average question length: {df['questions'].str.len().mean():.0f} characters")
print(f"Average answer length: {df['answers'].str.len().mean():.0f} characters")

# Categorize questions
categories = {
    'Greetings': ['hello', 'hi', 'hey', 'what\'s up'],
    'Company Info': ['who', 'what is apexify', 'vision', 'mission', 'different', 'about'],
    'Contact': ['contact', 'reach'],
    'Internships': ['internship', 'intern'],
    'Services': ['services', 'it services', 'skills development', 'consulting'],
    'Tech Info': ['artificial intelligence', 'ai', 'machine learning', 'data science', 'frontend', 'backend', 'digital marketing', 'graphic design'],
    'Process': ['apply', 'application', 'resume', 'paid', 'eligibility', 'skills', 'duration', 'remote', 'certificate', 'job', 'mentorship', 'projects', 'technologies'],
    'Misc': ['success stories', 'business hours']
}

print("\n📋 Category distribution:")
category_counts = {cat: 0 for cat in categories}
for q in df['questions']:
    q_lower = q.lower()
    for cat, keywords in categories.items():
        if any(keyword in q_lower for keyword in keywords):
            category_counts[cat] += 1
            break

for cat, count in category_counts.items():
    print(f"  {cat}: {count} questions")

# --- Download Files to Local Machine (if in Colab) ---
try:
    from google.colab import files
    print("\n" + "="*70)
    print("📥 DOWNLOADING FILES TO YOUR COMPUTER...")
    print("Click each link below to download:")

    files.download('vectorizer.pkl')
    files.download('tfidf_matrix.pkl')
    files.download('faq_data.pkl')
    files.download('apexify_faqs.csv')
    print("✅ Files downloaded! You can now use them in your web application.")
except ImportError:
    print("\n✅ Script finished. Not running in Google Colab, so files are saved locally in the current directory:")
    print("   - vectorizer.pkl")
    print("   - tfidf_matrix.pkl")
    print("   - faq_data.pkl")
    print("   - apexify_faqs.csv")

print("\n" + "="*70)
print("🎉 SUCCESS! The FAQ data is perfectly aligned with the Apexify Technologys website.")
print("="*70)

NLTK data downloaded.
Questions count: 52
Answers count: 52
✅ Created 52 FAQ pairs.
✅ FAQ data saved to 'apexify_faqs.csv'.
✅ Text preprocessing complete.
✅ TF-IDF matrix created with shape: (52, 128)
✅ Vocabulary size: 128

✅ All models and data saved successfully:
   - vectorizer.pkl
   - tfidf_matrix.pkl
   - faq_data.pkl
   - apexify_faqs.csv

TESTING THE CHATBOT WITH SAMPLE QUESTIONS

QUESTION                                 SCORE      MATCHED QUESTION              
--------------------------------------------------------------------------------
Hello there                              1.000     Hello                         
  Answer: Hello! Welcome to Apexify Technologys. How can I help you with your career journey today?...

What internships do you have?            0.540     Do you offer internships?     
  Answer: Yes, absolutely! We offer a comprehensive internship program for students and recent graduates looking to kickstart their careers in tech. You can app...

Tell me ab

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files downloaded! You can now use them in your web application.

🎉 SUCCESS! The FAQ data is perfectly aligned with the Apexify Technologys website.
